# 03 — Volatility Forecasting

**Environment**: Kaggle (GPU T4 / P100). Runs the full volatility pipeline via DVC.

**Prerequisites**: Notebook 02 must have been run and all its outputs pushed to the DVC remote.

**Pipeline stages executed** (in order):
1. `run_experiment` — GARCH + sentiment-LSTM hybrid forecasting
2. `run_robustness` — robustness checks across specifications
3. `plot_results` — generate forecast comparison figures
4. `plot_results_extras` — generate sentiment distribution & robustness figures
5. `descriptive_stats` — generate LaTeX descriptive stats table

**Expected outputs after this notebook**:
| Artifact | Path |
|---|---|
| LSTM model weights | `models/volatility/lstm_residual_model.pt` |
| Experiment results | `data/processed/hybrid_experiment_summary.json` |
| Robustness results | `data/processed/robustness_experiment_summary.json` |
| Final processed dataset | `data/processed/final_modeling_dataset.parquet` |
| Modeling frame | `data/interim/modeling_ready.parquet` |
| Report figures | `report/figures/*.png` |
| Descriptive stats | `report/tables/descriptive_stats_table.tex` |

In [ ]:
# ── 1. Environment Setup ────────────────────────────────────────────────────
import os, subprocess, sys
from pathlib import Path

REPO_URL    = "https://github.com/tlong-ds/news-sentiment-analysis.git"
REPO_BRANCH = "main"
REPO_DIR    = Path("/kaggle/working/news-sentiment-analysis")
ON_KAGGLE   = Path("/kaggle").exists()

def run(cmd: list[str], cwd: Path = REPO_DIR) -> None:
    subprocess.run(cmd, cwd=cwd, check=True)
    print(f"✓ {' '.join(str(c) for c in cmd)}")

if ON_KAGGLE:
    if not REPO_DIR.exists():
        run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
             REPO_URL, str(REPO_DIR)], cwd=Path("/kaggle/working"))
    run([sys.executable, "-m", "pip", "install", "-q", "-r",
         str(REPO_DIR / "requirements.txt")])
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
else:
    REPO_DIR = Path().resolve()

VOLATILITY_PIPELINE = REPO_DIR / "pipelines" / "volatility"

print(f"ROOT: {REPO_DIR}")

In [ ]:
# ── 2. DVC Remote Configuration ─────────────────────────────────────────────
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient  # noqa: import
    sa_json = UserSecretsClient().get_secret("GDRIVE_SERVICE_ACCOUNT_JSON")
    sa_path = REPO_DIR / ".dvc" / "gdrive_sa.json"
    sa_path.write_text(sa_json)
    run(["dvc", "remote", "modify", "gdrive",
         "gdrive_use_service_account", "true"])
    run(["dvc", "remote", "modify", "gdrive",
         "gdrive_service_account_json_file_path", str(sa_path)])
    print("DVC remote configured.")

In [ ]:
# ── 3. Pull upstream outputs from Notebook 02 ───────────────────────────────
# Pull from the repo root so DVC resolves the full stage graph and cache.
run(["dvc", "pull", "--run-cache", "-q"], cwd=REPO_DIR)


In [ ]:
# ── 4. Verify upstream artifacts are present ─────────────────────────────────
# After dvc pull, all upstream artifacts (preprocessing, sentiment, model frame)
# should already be present. Assert the key inputs before running the pipeline.
import yaml

params  = yaml.safe_load((REPO_DIR / "params.yaml").read_text())
interim = REPO_DIR / params["paths"]["interim_dir"]

required = [
    interim / "modeling_ready.parquet",
    REPO_DIR / params["paths"]["sentiment_dir"] / "article_sentiment_scores.parquet",
    REPO_DIR / params["paths"]["prices"],
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Required upstream artifacts are missing after dvc pull:\n"
        + "\n".join(f"  {m}" for m in missing)
        + "\nEnsure Notebook 02 has been run and its outputs pushed to the DVC remote."
    )
print("All required upstream artifacts are present.")
for p in required:
    print(f"  ✓ {p.relative_to(REPO_DIR)}")

In [ ]:
# ── 5. Run Full Volatility Pipeline ─────────────────────────────────────────
#
# We use `dvc repro --downstream` starting from `build_model_frame` (the last
# upstream sentinel stage whose outputs are already pulled from the DVC remote).
#
# `--downstream <stage>` tells DVC to run <stage> and ONLY the stages that
# depend on it (i.e., the volatility stages), WITHOUT traversing upstream
# deps (preprocess → prepare_training_data → bootstrap_labels → ...).
# This prevents re-triggering the sentiment pipeline which requires a Gemini
# API key and GPU training that were already done in Notebooks 01 and 02.
#
# `build_model_frame` output (modeling_ready.parquet) is up-to-date after
# the pull above, so DVC will skip it and proceed to the volatility stages.
run(
    ["dvc", "repro", "--downstream",
     "../sentiment/dvc.yaml:build_model_frame"],
    cwd=VOLATILITY_PIPELINE,
)

In [ ]:
# ── 6. Push All Outputs ──────────────────────────────────────────────────────
run(["dvc", "push"], cwd=VOLATILITY_PIPELINE)
print("All volatility outputs pushed to DVC remote.")

In [ ]:
# ── 7. Verify Outputs ────────────────────────────────────────────────────────
import json, yaml, pandas as pd
from pathlib import Path

params  = yaml.safe_load((REPO_DIR / "params.yaml").read_text())
interim = REPO_DIR / params["paths"]["interim_dir"]
vol     = params["volatility"]

# Modeling frame
mf = pd.read_parquet(interim / "modeling_ready.parquet")
print(f"modeling_ready.parquet (merged input): {len(mf):,} rows | {mf['date'].min()} → {mf['date'].max()}")

# Final processed dataset
fd_path = REPO_DIR / vol["output_final_dataset"]
fd = pd.read_parquet(fd_path)
print(f"final_modeling_dataset.parquet (processed data): {len(fd):,} rows | {fd['date'].min()} → {fd['date'].max()}")

# LSTM model
lstm_path = REPO_DIR / vol["output_lstm_model"]
print(f"LSTM model ({lstm_path.parent.name}/{lstm_path.name}): {lstm_path.stat().st_size / 1024:.1f} KB")

# Experiment results
exp = json.loads((REPO_DIR / vol["output_experiment"]).read_text())
print("\n=== Hybrid Experiment Results ===")
print(f"  Baseline RMSE : {exp['baseline_metrics']['rmse']:.6f}")
print(f"  Hybrid   RMSE : {exp['hybrid_metrics']['rmse']:.6f}")
dm = exp.get("diebold_mariano", {})
print(f"  DM test p-value: {dm.get('p_value', 'N/A')} (significant: {dm.get('significant_95', 'N/A')})")

# Robustness results
rob = json.loads((REPO_DIR / vol["output_robustness"]).read_text())
print("\n=== Robustness Specifications ===")
for spec_key, spec_val in rob.items():
    if isinstance(spec_val, dict) and "baseline_rmse" in spec_val:
        print(f"  {spec_key:30s} baseline={spec_val['baseline_rmse']:.6f}  hybrid={spec_val['hybrid_rmse']:.6f}")

# Report figures
figures = list((REPO_DIR / "report" / "figures").glob("*.png"))
print(f"\nFigures generated: {[f.name for f in figures]}")
tex = REPO_DIR / "report" / "tables" / "descriptive_stats_table.tex"
print(f"LaTeX table      : {tex.name} ({tex.stat().st_size} bytes)")